[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/mp-1/blob/master/notebooks/03_hooke_jeeves.ipynb)

# 03. Метод Хука-Дживса

Используем исследующий поиск и метод деления отрезка пополам для поиска шага.

In [ ]:
!pip install -q numpy pandas sympy matplotlib

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Path("figures").mkdir(exist_ok=True)
Path("tables").mkdir(exist_ok=True)

Q = np.array([[4.0, -2.0], [-2.0, 6.0]])
B = np.array([-8.0, 6.0])
C = 30.0
X0 = np.array([0.0, 0.0])
EPS = 1e-4
X_STAR = np.linalg.solve(Q, -B)
F_STAR = 21.6

def f(point):
    x = np.asarray(point, dtype=float)
    return float(0.5 * x @ Q @ x + B @ x + C)

def grad(point):
    x = np.asarray(point, dtype=float)
    return Q @ x + B

def original_formula(x, y):
    return 2 * (x - 3) ** 2 + 3 * (y + 2) ** 2 - 2 * x * y + 4 * x - 6 * y

def grid():
    x = np.linspace(-1.0, 4.0, 220)
    y = np.linspace(-3.0, 2.0, 220)
    xx, yy = np.meshgrid(x, y)
    return xx, yy, original_formula(xx, yy)

def plot_path(points, title, simplexes=None, out=None):
    xx, yy, zz = grid()
    fig, ax = plt.subplots(figsize=(7, 5), dpi=130)
    contour = ax.contour(xx, yy, zz, levels=25, cmap="viridis")
    ax.clabel(contour, inline=True, fontsize=7)
    if simplexes is not None:
        for simplex in simplexes:
            closed = np.vstack([simplex, simplex[0]])
            ax.plot(closed[:, 0], closed[:, 1], color="#4b5563", alpha=0.35, linewidth=1)
    points = np.asarray(points)
    ax.plot(points[:, 0], points[:, 1], marker="o", linewidth=1.8, markersize=3.5, label="траектория")
    ax.scatter([points[0, 0]], [points[0, 1]], color="#2563eb", s=45, label="x0")
    ax.scatter([X_STAR[0]], [X_STAR[1]], color="red", marker="*", s=120, label="x*")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    if out:
        fig.savefig(out, bbox_inches="tight")
    plt.show()

In [ ]:
def bisection_min(func, a, b, tol=1e-4):
    while (b - a) > tol:
        mid = (a + b) / 2.0
        left = a + (b - a) / 4.0
        right = b - (b - a) / 4.0
        f_left, f_mid, f_right = func(left), func(mid), func(right)
        if f_left < f_mid:
            b = mid
        elif f_right < f_mid:
            a = mid
        else:
            a, b = left, right
    t = (a + b) / 2.0
    return t, float(func(t))

def exploratory_search(func, base, delta):
    point = base.copy()
    for i in range(len(point)):
        best_value = func(point)
        trial = point.copy()
        trial[i] += delta
        if func(trial) < best_value:
            point = trial
            continue
        trial = point.copy()
        trial[i] -= delta
        if func(trial) < best_value:
            point = trial
    return point

def hooke_jeeves(func, x0, delta=1.0, tol=1e-4, max_iter=300):
    base = np.asarray(x0, dtype=float)
    history = []
    iteration = 0
    while delta > tol and iteration < max_iter:
        explored = exploratory_search(func, base, delta)
        if func(explored) < func(base):
            direction = explored - base
            phi = lambda t: func(explored + t * direction)
            t_star, _ = bisection_min(phi, 0.0, 2.0, tol)
            candidate = explored + t_star * direction
            base = candidate if func(candidate) < func(base) else explored
        else:
            delta *= 0.5
        history.append({"iter": iteration, "x": base.copy(), "f": float(func(base)), "delta": float(delta)})
        iteration += 1
    return base, float(func(base)), history

In [ ]:
x_min, f_min, history = hooke_jeeves(f, X0, tol=EPS)
print("x =", x_min, "f =", f_min, "iterations =", len(history))

rows = []
for item in history[:8] + [history[-1]]:
    x = item["x"]
    rows.append([item["iter"], x[0], x[1], item["f"], item["delta"]])
df = pd.DataFrame(rows, columns=["k", "x_k", "y_k", "f(x_k)", "delta"])
df.to_csv("tables/tab_4_2_hj.csv", index=False)
display(df)

plot_path(np.array([item["x"] for item in history]), "Метод Хука-Дживса", out="figures/fig_4_2_hj_trajectory.png")